In [1]:
from pyomo.environ import *
from pyomo.gdp import Disjunct, Disjunction

In [2]:
model = ConcreteModel()

In [3]:
products = ["E", "A", "B"]
units = ["D_Column", "M_Unit", "A_Unit"]

In [4]:
model.profit = Param(units, products, initialize={
    ("D_Column", "E"): 60, ("D_Column", "A"): 45, ("D_Column", "B"): 80,
    ("M_Unit", "E"): 55, ("M_Unit", "A"): 50, ("M_Unit", "B"): 70,
    ("A_Unit", "E"): 65, ("A_Unit", "A"): 40, ("A_Unit", "B"): 75,
})
model.energy = Param(units, products, initialize={
    ("D_Column", "E"): 3, ("D_Column", "A"): 2, ("D_Column", "B"): 4,
    ("M_Unit", "E"): 1, ("M_Unit", "A"): 1.5, ("M_Unit", "B"): 2,
    ("A_Unit", "E"): 2, ("A_Unit", "A"): 2, ("A_Unit", "B"): 3,
})

model.feedLimit = Param(initialize=100)
model.demandLimit = Param(initialize=30)
model.contractLimit = Param(initialize=10)
model.fixedCost = Param(units, initialize={"D_Column": 500, "M_Unit": 300, "A_Unit":400})
model.energyAvailable = Param(units, initialize={"D_Column": 250, "M_Unit": 150, "A_Unit":200})

In [5]:
model.make = Var(products, within=NonNegativeReals, bounds=(0, 100))
model.Profit = Var(within=Reals, bounds=(-500, 7000))

In [6]:
model.MaximizeProfit = Objective(expr = model.Profit, sense = maximize)

In [7]:
model.FeedConstraint = Constraint(expr = sum(model.make[i] for i in products) <= model.feedLimit)
model.DemandConstraint = Constraint(expr = model.make["B"] <= model.demandLimit)
model.contractConstraint = Constraint(expr = model.make["E"] >= model.contractLimit)

In [ ]:
model.U = Disjunct(units)

for u in units:
    model.U[u].ProfitConstraint = Constraint(
        expr = model.Profit == sum(model.profit[u, i]*model.make[i] for i in products) - model.fixedCost[u])
    model.U[u].EnergyConstraint = Constraint(
        expr = sum(model.energy[u, i]*model.make[i] for i in products) <= model.energyAvailable[u])

In [ ]:
model.unitChoice = Disjunction(expr = [model.U[u] for u in units], xor=True)

In [10]:
TransformationFactory("gdp.bigm").apply_to(model)

In [11]:
optimizer = SolverFactory("appsi_highs")
optimizer.solve(model)
model.display()

Model unknown

  Variables:
    make : Size=3, Index={E, A, B}
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          A :     0 :   0.0 :   100 : False : False : NonNegativeReals
          B :     0 :  -0.0 :   100 : False : False : NonNegativeReals
          E :     0 : 100.0 :   100 : False : False : NonNegativeReals
    Profit : Size=1, Index=None
        Key  : Lower : Value             : Upper : Fixed : Stale : Domain
        None :  -500 : 6100.000000000003 :  7000 : False : False :  Reals

  Objectives:
    MaximizeProfit : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 6100.000000000003

  Constraints:
    FeedConstraint : Size=1
        Key  : Lower : Body  : Upper
        None :  None : 100.0 : 100.0
    DemandConstraint : Size=1
        Key  : Lower : Body : Upper
        None :  None : -0.0 :  30.0
    contractConstraint : Size=1
        Key  : Lower : Body  : Upper
        None :  10.0 : 100.0 :  None

  Blocks:
    Bl